# SLV_TRANSFORM_TRANSACTIONS
**Layer:** Silver  
**Purpose:** Incremental load from Bronze `brz_transaction_logs` → apply data quality checks → upsert to Silver Delta table `slv_transaction_logs`.  
**Pattern:** Read-new-watermark → DQ → transform → MERGE into Delta.

## 1. Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline parameters — injected by Synapse Pipeline at runtime.
# ---------------------------------------------------------------------------
batch_id             = "dev-run-00000000"
storage_account      = "adlsbankingdev"
bronze_container     = "bronze"
silver_container     = "silver"
keyvault_name        = "kv-banking-dev"
watermark_date       = "2024-01-01"   # Lower bound: ingestion_date >= watermark_date
run_date             = "2024-01-02"   # Upper bound: ingestion_date <  run_date (exclusive)
dq_reject_threshold  = 0.05           # Abort if rejected fraction exceeds 5 %

## 2. Imports and Spark Configuration

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DecimalType, TimestampType, DoubleType
)
from delta.tables import DeltaTable
import datetime

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

def adls_path(container, *parts):
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return "/".join([base] + list(parts))

ingestion_timestamp = datetime.datetime.utcnow().isoformat() + "Z"
print(f"batch_id={batch_id}  watermark={watermark_date}  run_date={run_date}")
print(f"Silver base: {adls_path(silver_container)}")

## 3. Read Bronze — Incremental Window

In [ ]:
bronze_path = adls_path(bronze_container, "raw", "brz_transaction_logs")

brz_df = (
    spark.read
    .format("parquet")
    .load(bronze_path)
    .filter(
        (F.col("ingestion_date") >= F.lit(watermark_date)) &
        (F.col("ingestion_date") <  F.lit(run_date))       &
        (F.col("source_system")  == F.lit("transaction_logs"))
    )
)

raw_count = brz_df.cache().count()
print(f"Bronze records in window [{watermark_date}, {run_date}): {raw_count:,}")

## 4. Parse Raw JSON Payload

In [ ]:
# Inline schema for the raw JSON body of transaction log events
txn_raw_schema = StructType([
    StructField("raw_log_id",          StringType(),  False),
    StructField("account_id",          StringType(),  True),
    StructField("channel_code",        StringType(),  True),
    StructField("txn_type_code",       StringType(),  True),
    StructField("service_code",        StringType(),  True),
    StructField("transaction_amount",  DoubleType(),  True),
    StructField("currency_code",       StringType(),  True),
    StructField("status_code",         StringType(),  True),
    StructField("response_time_ms",    LongType(),    True),
    StructField("event_timestamp",     StringType(),  True),   # ISO 8601 string
    StructField("merchant_id",         StringType(),  True),
    StructField("error_code",          StringType(),  True),
])

parsed_df = (
    brz_df
    .withColumn("payload", F.from_json(F.col("raw_payload"), txn_raw_schema))
    .select(
        "payload.*",
        "ingestion_date",
        "enqueued_time",
        "batch_id",
        "source_system"
    )
)

## 5. Data Quality Checks

In [ ]:
# ── 5a. Null checks on mandatory fields ──────────────────────────────────────
mandatory_fields = ["raw_log_id", "account_id", "transaction_amount", "status_code", "event_timestamp"]

null_conditions = [F.col(f).isNull() for f in mandatory_fields]
combined_null   = null_conditions[0]
for cond in null_conditions[1:]:
    combined_null = combined_null | cond

null_df     = parsed_df.filter(combined_null)
non_null_df = parsed_df.filter(~combined_null)
null_count  = null_df.count()

# ── 5b. Duplicate removal on raw_log_id (keep first by enqueued_time) ────────
from pyspark.sql.window import Window

dedup_window  = Window.partitionBy("raw_log_id").orderBy(F.col("enqueued_time").asc())
ranked_df     = non_null_df.withColumn("_rn", F.row_number().over(dedup_window))
duplicate_df  = ranked_df.filter(F.col("_rn") > 1)
deduped_df    = ranked_df.filter(F.col("_rn") == 1).drop("_rn")
duplicate_count = duplicate_df.count()

# ── 5c. Amount range validation (amount must be > 0) ─────────────────────────
invalid_amount_df = deduped_df.filter(F.col("transaction_amount") <= 0)
valid_df          = deduped_df.filter(F.col("transaction_amount") > 0)
invalid_amount_count = invalid_amount_df.count()

# ── 5d. Aggregate rejection summary ──────────────────────────────────────────
rejected_count = null_count + duplicate_count + invalid_amount_count
reject_fraction = rejected_count / raw_count if raw_count > 0 else 0.0

print(f"DQ Summary  raw={raw_count:,}  null={null_count:,}  duplicate={duplicate_count:,}  "
      f"bad_amount={invalid_amount_count:,}  total_rejected={rejected_count:,}  "
      f"reject_pct={reject_fraction*100:.2f}%")

if reject_fraction > dq_reject_threshold:
    raise RuntimeError(
        f"DQ threshold exceeded: {reject_fraction*100:.2f}% rejected > {dq_reject_threshold*100:.0f}% threshold. "
        "Aborting batch."
    )

## 6. Apply Silver Schema Transformations

In [ ]:
# ISO 4217 currency normalisation mapping (extend as needed)
currency_map = {
    "USD": "USD", "US$": "USD", "usd": "USD",
    "EUR": "EUR", "GBP": "GBP", "INR": "INR",
    "SGD": "SGD", "AUD": "AUD",
}
currency_map_expr = F.create_map(*[x for kv in currency_map.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

# Status code standardisation
status_map_expr = F.create_map(
    F.lit("S"),       F.lit("SUCCESS"),
    F.lit("SUCCESS"), F.lit("SUCCESS"),
    F.lit("F"),       F.lit("FAILURE"),
    F.lit("FAIL"),    F.lit("FAILURE"),
    F.lit("FAILURE"), F.lit("FAILURE"),
    F.lit("P"),       F.lit("PENDING"),
    F.lit("PENDING"), F.lit("PENDING"),
    F.lit("T"),       F.lit("TIMEOUT"),
    F.lit("TIMEOUT"), F.lit("TIMEOUT"),
)

transformed_df = (
    valid_df
    .withColumn("event_timestamp",    F.to_timestamp(F.col("event_timestamp")))
    .withColumn("transaction_amount", F.col("transaction_amount").cast(DecimalType(18, 4)))
    .withColumn("response_time_ms",   F.col("response_time_ms").cast(LongType()))
    # Derive integer date surrogate key YYYYMMDD
    .withColumn("event_date_sk",
        F.date_format(F.col("event_timestamp"), "yyyyMMdd").cast(IntegerType()))
    # Standardise status
    .withColumn("status_code_std",
        F.coalesce(status_map_expr[F.upper(F.col("status_code"))], F.upper(F.col("status_code"))))
    # Normalise currency
    .withColumn("currency_iso",
        F.coalesce(currency_map_expr[F.upper(F.col("currency_code"))], F.upper(F.col("currency_code"))))
    # Audit columns
    .withColumn("slv_batch_id",             F.lit(batch_id))
    .withColumn("slv_ingestion_timestamp",  F.lit(ingestion_timestamp))
    .withColumnRenamed("status_code_std",   "status_code")
    .withColumnRenamed("currency_iso",      "currency_code")
    .drop("source_system", "ingestion_date")
)

## 7. Surrogate Key Lookups (Broadcast Joins)

In [ ]:
# Read dimension tables for surrogate key resolution.
# Broadcast hints ensure map-side joins — no shuffle for small dims.
silver_path = adls_path(silver_container, "delta")

dim_account = (
    spark.read.format("delta")
    .load(f"{silver_path}/slv_dim_account")
    .select("account_id", "account_sk")
)

dim_channel = (
    spark.read.format("delta")
    .load(f"{silver_path}/slv_dim_channel")
    .select("channel_code", "channel_sk")
)

dim_txn_type = (
    spark.read.format("delta")
    .load(f"{silver_path}/slv_dim_transaction_type")
    .select("txn_type_code", "txn_type_sk")
)

enriched_df = (
    transformed_df
    .join(F.broadcast(dim_account),   on="account_id",    how="left")
    .join(F.broadcast(dim_channel),   on="channel_code",  how="left")
    .join(F.broadcast(dim_txn_type),  on="txn_type_code", how="left")
)
print(f"Records after surrogate key enrichment: {enriched_df.count():,}")

## 8. Upsert to Silver Delta Table (MERGE)

In [ ]:
silver_table_path = f"{silver_path}/slv_transaction_logs"

# Bootstrap: create the table on first run if it does not exist
if not DeltaTable.isDeltaTable(spark, silver_table_path):
    print("Target Delta table not found — creating from current batch.")
    (
        enriched_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_date_sk")
        .save(silver_table_path)
    )
    print(f"Delta table created at {silver_table_path}")
else:
    silver_delta = DeltaTable.forPath(spark, silver_table_path)

    (
        silver_delta.alias("target")
        .merge(
            enriched_df.alias("source"),
            "target.raw_log_id = source.raw_log_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    op_metrics = (
        silver_delta.history(1)
        .select("operationMetrics")
        .collect()[0][0]
    )
    print(f"MERGE complete. Metrics: {op_metrics}")

## 9. Log DQ Metrics to Control Table

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType as TSType
import datetime

dq_schema = StructType([
    StructField("batch_id",          StringType(), False),
    StructField("pipeline_stage",    StringType(), False),
    StructField("source_table",      StringType(), False),
    StructField("target_table",      StringType(), False),
    StructField("run_ts",            StringType(), False),
    StructField("raw_count",         LongType(),   True),
    StructField("null_count",        LongType(),   True),
    StructField("duplicate_count",   LongType(),   True),
    StructField("rejected_count",    LongType(),   True),
    StructField("loaded_count",      LongType(),   True),
    StructField("reject_fraction",   DoubleType(), True),
])

dq_row = [(
    batch_id,
    "slv_transform_transactions",
    "brz_transaction_logs",
    "slv_transaction_logs",
    ingestion_timestamp,
    int(raw_count),
    int(null_count),
    int(duplicate_count),
    int(rejected_count),
    int(enriched_df.count()),
    float(reject_fraction),
)]

dq_df = spark.createDataFrame(dq_row, schema=dq_schema)

ctrl_path = adls_path(silver_container, "delta", "slv_ctrl_dq_metrics")
(
    dq_df.write
    .format("delta")
    .mode("append")
    .save(ctrl_path)
)
print(f"DQ metrics written to control table: {ctrl_path}")
print(f"batch_id={batch_id}  loaded={enriched_df.count():,}  rejected={rejected_count:,}")